# ArUco Marker Erstellung und Kamerakalibrierung

Dieses Notebook dient zur Generierung von ArUco-Markern sowie zur Durchführung der Kamerakalibrierung für das ArUco-Tanks-Spielsystem.

## Inhalte
- Generierung individueller Marker
- Erstellung eines Kalibrierungsboards
- Aufnahme von Kalibrierungsbildern
- Berechnung der Kameramatrix

## Voraussetzungen
- Python ≥ 3.9
- OpenCV
- NumPy
- Webcam


In [ ]:
import cv2
from cv2 import aruco
import numpy as np

# Dictionary auswählen
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

# Marker erzeugen
marker_id = 0
marker_size = 200
marker = aruco.generateImageMarker(aruco_dict, marker_id, marker_size)

# Anzeigen
import matplotlib.pyplot as plt
plt.imshow(marker, cmap="gray")
plt.axis("off")


In [ ]:
import cv2
from cv2 import aruco
import numpy as np

# Dictionary auswählen
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

# Marker erzeugen
marker_id = 1
marker_size = 200
marker = aruco.generateImageMarker(aruco_dict, marker_id, marker_size)

# Anzeigen
import matplotlib.pyplot as plt
plt.imshow(marker, cmap="gray")
plt.axis("off")

In [ ]:
import cv2
import numpy as np
from cv2 import aruco

# A4 bei 300 DPI
dpi = 300
a4_w_mm, a4_h_mm = 210, 297
a4_w_px = int(a4_w_mm / 25.4 * dpi)
a4_h_px = int(a4_h_mm / 25.4 * dpi)

# Marker-Settings
ids = [0, 1]
border_frac = 0.25   # 25% Rand je Seite
gap_px = 180         # Abstand zwischen den Markern
outer_margin_px = 120  # Rand zum Papierrand

aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

def add_white_border(img, border_frac=0.25):
    h, w = img.shape[:2]
    b = int(min(h, w) * border_frac)
    out = np.ones((h + 2*b, w + 2*b), dtype=np.uint8) * 255
    out[b:b+h, b:b+w] = img
    return out

# Maximale Breite pro Marker (inkl. Rand) so, dass 2 Marker + gap + Außenrand auf A4 passen
usable_w = a4_w_px - 2*outer_margin_px - gap_px
max_outer_per_marker = usable_w // 2  # outer = marker + 2*border
# outer = marker*(1 + 2*border_frac)  => marker = outer / (1+2*border_frac)
marker_size_px = int(max_outer_per_marker / (1 + 2*border_frac))

# Marker erzeugen
markers = []
for mid in ids:
    m = aruco.generateImageMarker(aruco_dict, mid, marker_size_px)
    m = add_white_border(m, border_frac)
    markers.append(m)

m0, m1 = markers
h0, w0 = m0.shape
h1, w1 = m1.shape

# A4 Leinwand
page = np.ones((a4_h_px, a4_w_px), dtype=np.uint8) * 255

# Zentrieren
total_w = w0 + gap_px + w1
x0 = (a4_w_px - total_w) // 2
y0 = (a4_h_px - max(h0, h1)) // 2

# Einfügen
page[y0:y0+h0, x0:x0+w0] = m0
page[y0:y0+h1, x0+w0+gap_px:x0+w0+gap_px+w1] = m1

out_name = f"aruco_A4_ids{ids[0]}_{ids[1]}_{dpi}dpi.png"
cv2.imwrite(out_name, page)

print("A4 width px:", a4_w_px, "| computed marker_size_px:", marker_size_px, "| outer marker px:", w0)
print("Saved:", out_name)


In [ ]:
import cv2
from cv2 import aruco
import numpy as np

# A4 bei 300 DPI
dpi = 300
a4_width_mm = 210
a4_height_mm = 297

a4_width_px = int(a4_width_mm / 25.4 * dpi)
a4_height_px = int(a4_height_mm / 25.4 * dpi)

# ChArUco-Parameter
squares_x = 5
squares_y = 7
square_length_mm = 30
marker_length_mm = 22

square_length_m = square_length_mm / 1000
marker_length_m = marker_length_mm / 1000

aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
board = aruco.CharucoBoard(
    (squares_x, squares_y),
    square_length_m,
    marker_length_m,
    aruco_dict
)

# Boardgröße in Pixeln
board_width_mm = squares_x * square_length_mm
board_height_mm = squares_y * square_length_mm

board_width_px = int(board_width_mm / 25.4 * dpi)
board_height_px = int(board_height_mm / 25.4 * dpi)

board_img = board.generateImage((board_width_px, board_height_px))

# Weißes A4-Blatt
a4 = np.ones((a4_height_px, a4_width_px), dtype=np.uint8) * 255

# Board zentrieren
x0 = (a4_width_px - board_width_px) // 2
y0 = (a4_height_px - board_height_px) // 2
a4[y0:y0+board_height_px, x0:x0+board_width_px] = board_img

cv2.imwrite("charuco_A4_5x7_30mm.png", a4)
print("Gespeichert: charuco_A4_5x7_30mm.png")


In [ ]:
import cv2
import os
import pygame

save_dir = "calib_images"
os.makedirs(save_dir, exist_ok=True)

cap = cv2.VideoCapture(0)

# Pygame für Controller initialisieren
pygame.init()
pygame.joystick.init()

joy = None
A_BUTTON = 0  # Xbox A ist häufig Button 0

if pygame.joystick.get_count() > 0:
    joy = pygame.joystick.Joystick(0)
    joy.init()
    print("Gamepad verbunden:", joy.get_name())
else:
    print("Kein Gamepad gefunden. Nur Tastatur 's' möglich.")

count = 0
a_was_pressed = False  # damit Halten nicht mehrfach speichert

while True:
    ret, frame = cap.read()
    if not ret:
        break

    text = f"Saved: {count}  (Press 's' or Xbox A to save, ESC to quit)"
    cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 3)
    cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 1)

    cv2.imshow("Capture calibration images", frame)

    # Pygame Events pumpen, damit Controllerstatus aktualisiert wird
    pygame.event.pump()

    save_triggered = False

    # Tastatur über OpenCV
    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC
        break
    if key == ord('s'):
        save_triggered = True

    # Controller A-Button
    if joy is not None:
        try:
            a_pressed = bool(joy.get_button(A_BUTTON))
            if a_pressed and not a_was_pressed:
                save_triggered = True
            a_was_pressed = a_pressed
        except pygame.error:
            pass

    if save_triggered:
        fname = os.path.join(save_dir, f"img_{count:03d}.png")
        cv2.imwrite(fname, frame)
        count += 1
        print("Saved", fname)

cap.release()
cv2.destroyAllWindows()
pygame.quit()

In [ ]:
import cv2
from cv2 import aruco
import numpy as np
import glob

# Board-Definition muss exakt zu deinem Ausdruck passen:
squares_x = 5
squares_y = 7
square_length_m = 0.0292
marker_length_m = 0.0214
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
board = aruco.CharucoBoard((squares_x, squares_y), square_length_m, marker_length_m, aruco_dict)

params = aruco.DetectorParameters()
detector = aruco.ArucoDetector(aruco_dict, params)

image_files = sorted(glob.glob("calib_images/*.png"))
if len(image_files) < 10:
    raise RuntimeError("Zu wenige Bilder. Nimm idealerweise 15–30 auf.")

all_charuco_corners = []
all_charuco_ids = []
image_size = None

for f in image_files:
    img = cv2.imread(f)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if image_size is None:
        image_size = gray.shape[::-1]  # (width, height)

    corners, ids, _ = detector.detectMarkers(gray)

    if ids is None or len(ids) < 4:
        continue

    # Optional: Refinement (hilft manchmal)
    # aruco.refineDetectedMarkers(gray, board, corners, ids, rejected=None)

    ok, charuco_corners, charuco_ids = aruco.interpolateCornersCharuco(
        markerCorners=corners,
        markerIds=ids,
        image=gray,
        board=board
    )

    if ok and charuco_corners is not None and len(charuco_corners) >= 10:
        all_charuco_corners.append(charuco_corners)
        all_charuco_ids.append(charuco_ids)

print("Verwendete Frames:", len(all_charuco_corners))

if len(all_charuco_corners) < 8:
    raise RuntimeError("Zu wenige brauchbare Frames. Mehr/bessere Aufnahmen nötig.")

# Kalibrierung
retval, camera_matrix, dist_coeffs, rvecs, tvecs = aruco.calibrateCameraCharuco(
    charucoCorners=all_charuco_corners,
    charucoIds=all_charuco_ids,
    board=board,
    imageSize=image_size,
    cameraMatrix=None,
    distCoeffs=None
)

print("Reprojection error:", retval)
print("Camera matrix:\n", camera_matrix)
print("Dist coeffs:\n", dist_coeffs.ravel())

np.savez("camera_calib_neu.npz", camera_matrix=camera_matrix, dist_coeffs=dist_coeffs, image_size=image_size)
print("Gespeichert: camera_calib.npz")


In [ ]:
import numpy as np
import cv2

data = np.load("camera_calib_neu.npz")
camera_matrix = data["camera_matrix"]
dist_coeffs = data["dist_coeffs"]

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break

    undist = cv2.undistort(frame, camera_matrix, dist_coeffs)
    cv2.imshow("Original", frame)
    cv2.imshow("Undistorted", undist)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
